# CycleGAN melon: detector arms for the EMA-session generators

Nine fixed parents (7 ripe, 2 unripe; only the disputed DSC_0093 is excluded).
Arms: `real_only`, `geometry_balanced`, `cyclegan_g0_ema_raw` and
`cyclegan_g0x_ema_raw`. Both CycleGAN bundles come from notebook 08 (same session,
full 60+60 schedule with AMP, raw final weights) and were accepted by a named blind
review with coverage of all nine parents. The EMA-weight bundles are not used:
the G0 EMA bundle does not cover DSC_0222.

Frozen detector protocol: yolov8n, 760 optimizer updates, batch 8, lr 1e-4,
letterbox 640, horizontal flip only, seeds 0-4, validation split only. The test
split is never opened. Two arms per GPU on T4 x2; the seeds stay paired across arms.

In [ ]:
# The input mounts are read only and ultralytics writes next to its working
# directory, so the whole code tree is copied into /kaggle/working first.
import os, shutil, subprocess, sys, time
from pathlib import Path

INPUT = Path('/kaggle/input')
print('mounted:', sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else 'nothing')

def find(*names, marker=None):
    # Kaggle mounts a dataset at /kaggle/input/<slug> or, when several are
    # attached, at /kaggle/input/datasets/<owner>/<slug>, so the slug is searched
    # by name at any depth before falling back to the marker file.
    for name in names:
        for candidate in [INPUT / name, *sorted(INPUT.rglob(name))]:
            if candidate.is_dir() and (marker is None or (candidate / marker).exists()):
                return candidate
    hits = sorted({p.parent for p in INPUT.rglob(marker)}) if marker else []
    if len(hits) == 1: return hits[0]
    raise SystemExit(f'cannot locate {names} (marker {marker}); found {hits}')

SOURCE = find('melon-cyclegan-code', marker='Ujicoba CycleGAN')
DATASET = find('melon-dataset-regrouped-v4', 'melon-final-reviewed-v4', marker='manifest.csv')
ASSETS = find('melon-benchmark-assets', marker='reviewed_masks.json')

WORK = Path('/kaggle/working')
CODE = WORK / 'code'
if CODE.exists(): shutil.rmtree(CODE)
shutil.copytree(SOURCE, CODE)
HERE = CODE / 'Ujicoba CycleGAN'
os.environ['MELON_DATASET'] = str(DATASET)
os.environ['MELON_ASSETS'] = str(ASSETS)
sys.path.insert(0, str(HERE))

import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('code:', sorted(p.name for p in HERE.iterdir()))
print('dataset:', (DATASET / 'manifest.csv').exists(), 'assets:', (ASSETS / 'reviewed_masks.json').exists())


def run(*command, cwd=None):
    started = time.time()
    result = subprocess.run([sys.executable, *[str(c) for c in command]], cwd=str(cwd or HERE))
    print(f'-> returncode {result.returncode} in {time.time() - started:.1f}s', flush=True)
    if result.returncode != 0: raise SystemExit(result.returncode)

In [ ]:
# The Kaggle image no longer ships ultralytics; the detector needs it. Nothing
# else is installed, so torch and torchvision stay at the image versions.
import importlib.util
if importlib.util.find_spec('ultralytics') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
import importlib.metadata as meta
for package in ('torch', 'torchvision', 'ultralytics'):
    try: print(package, meta.version(package))
    except meta.PackageNotFoundError: print(package, 'NOT INSTALLED')

In [ ]:
REVIEWED = find('melon-cyclegan-bundles-ema', marker='bundle_cyclegan_g0x_ema_raw')
EXCLUDE = ['DSC_0093']
SCOPE = ['DSC_0059', 'DSC_0069', 'DSC_0109', 'DSC_0110', 'DSC_0146', 'DSC_0222', 'DSC_0397', 'oke3510', 'oke5045']
BUNDLES = [REVIEWED / 'bundle_cyclegan_g0_ema_raw', REVIEWED / 'bundle_cyclegan_g0x_ema_raw']
import collections, json
for bundle in BUNDLES:
    records = json.loads((bundle / 'records.json').read_text())
    status = collections.Counter(r['status'] for r in records)
    print(bundle.name, dict(status), 'reviewers:', sorted({r.get('reviewer', '') for r in records}))
    if status.get('needs_review'): raise SystemExit(f'{bundle.name} still has unreviewed candidates')
    missing = sorted(set(SCOPE) - {r['parent_id'] for r in records if r['status'] == 'accepted'})
    if missing: raise SystemExit(f'{bundle.name} serves no accepted candidate for {missing}; scope differs')
ARMS = WORK / 'detector_arms_ema'
run(HERE / 'tools' / 'build_arms.py', '--bundle', *BUNDLES, '--arms', 'cyclegan_g0_ema_raw', 'cyclegan_g0x_ema_raw',
    '--out', ARMS, '--dataset', DATASET, '--balanced-geometry', '--exclude', *EXCLUDE)
summary = json.loads((ARMS / 'arms_summary.json').read_text())
assert set(summary['parents']) == set(SCOPE), 'scope differs'
for name, counts in summary['arms'].items(): print(' ', name, counts)

In [ ]:
import torch, subprocess
GROUPS = [['real_only', 'cyclegan_g0_ema_raw'], ['geometry_balanced', 'cyclegan_g0x_ema_raw']]
n_gpu = torch.cuda.device_count()
print('visible GPUs:', n_gpu)
def launch(arms, gpu, index):
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'PYTHONPATH': str(HERE)}
    log = open(WORK / f'detector_group{index}.log', 'w')
    command = [sys.executable, str(HERE / 'tools' / 'train_detector_arms.py'), '--arms-dir', str(ARMS),
               '--arms', *arms, '--out', str(WORK / f'detector_group{index}'), '--export']
    return subprocess.Popen(command, cwd=str(HERE), env=env, stdout=log, stderr=subprocess.STDOUT)
started = time.time()
if n_gpu >= 2:
    procs = [launch(arms, i, i) for i, arms in enumerate(GROUPS)]
    codes = [proc.wait() for proc in procs]
else:
    codes = [launch(arms, 0, i).wait() for i, arms in enumerate(GROUPS)]
print('return codes', codes, 'wall minutes', round((time.time() - started) / 60, 1))
for i in range(len(GROUPS)): print((WORK / f'detector_group{i}.log').read_text()[-2000:])
if any(codes): raise SystemExit('detector training failed')

In [ ]:
# One results file for all four arms, aggregated exactly as train_detector_arms.py does.
import statistics
OUT = WORK / 'detector_cyclegan_ema'
OUT.mkdir(exist_ok=True)
results, seconds, seeds = {}, 0.0, None
for i in range(len(GROUPS)):
    part = json.loads((WORK / f'detector_group{i}' / 'results.json').read_text())
    results.update(part['results']); seconds += part['seconds']; seeds = part['seeds']
    shutil.copytree(WORK / f'detector_group{i}', OUT / f'group{i}', dirs_exist_ok=True)
arms = ['real_only', 'geometry_balanced', 'cyclegan_g0_ema_raw', 'cyclegan_g0x_ema_raw']
rows = []
for arm in arms:
    keys = [f'{arm}@{s}' for s in seeds]
    values = [results[k]['map50_95'] for k in keys]
    rows.append(dict(arm=arm, runs=len(values), map50_95=round(statistics.fmean(values), 6),
                     sd=round(statistics.pstdev(values), 6),
                     ap_unripe=round(statistics.fmean(results[k]['per_class']['unripe']['ap50_95'] for k in keys), 6),
                     recall_unripe=round(statistics.fmean(results[k]['recall'][1] for k in keys), 6)))
base = rows[0]['map50_95']; geometry = rows[1]['map50_95']
for row in rows:
    row['delta_vs_real'] = round(row['map50_95'] - base, 6)
    row['delta_vs_geometry'] = round(row['map50_95'] - geometry, 6)
(OUT / 'results.json').write_text(json.dumps(dict(results=results, arms=arms, seeds=seeds, updates=760, batch=8,
    lr=1e-4, summary=rows, geometry_reference='geometry_balanced', seconds_sum_of_groups=round(seconds, 1),
    scope=SCOPE, test_used=False), indent=2) + '\n')
for row in rows: print(row)

In [ ]:
shutil.rmtree(CODE, ignore_errors=True)
shutil.rmtree(ARMS, ignore_errors=True)
for i in range(len(GROUPS)): shutil.rmtree(WORK / f'detector_group{i}', ignore_errors=True)
print(sorted(p.name for p in WORK.iterdir()))